# 02 · Preprocessing and evaluation

## Load data

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent

df = pd.read_csv(root / "data/raw/online_shoppers_intention.csv")

## Remove repeated rows before splitting

Keep one copy of each repeated row so identical records cannot appear in both sets. Without session IDs, some could be separate sessions. The raw CSV stays unchanged.

In [2]:
rows_before = len(df)
df = df.drop_duplicates()
pd.Series({
    "Original rows": rows_before,
    "Repeated rows removed": rows_before - len(df),
    "Rows kept": len(df),
})

Original rows            12330
Repeated rows removed      125
Rows kept                12205
dtype: int64

## Stratified 80/20 train/test split

Stratification keeps a similar share of purchases in both sets. Use `random_state=42` to repeat the same split. Keep the test set aside; preprocessing and model selection will use training data only.

In [3]:
X = df.drop(columns="Revenue")
X["Weekend"] = X["Weekend"].astype(int)
y = df["Revenue"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(df)

pd.DataFrame({
    "sessions": [len(y_train), len(y_test)],
    "purchases": [y_train.sum(), y_test.sum()],
    "purchase_percentage": [y_train.mean() * 100, y_test.mean() * 100],
}, index=["Train", "Test"]).round(2)

,sessions,purchases,purchase_percentage
Train,9764,1526,15.63
Test,2441,382,15.65


## Preprocessing pipeline

Scale numeric features for logistic regression and one-hot encode categories, including numeric category codes. `Weekend` is converted to 0/1 before splitting. There are no missing values, so imputation is not needed.

Keep preprocessing inside the model pipeline so it is fitted on each training fold, not the test set. Unknown categories are ignored when transforming new data.

In [4]:
numeric_features = [
    "Administrative", "Administrative_Duration", "Informational",
    "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay",
]
categorical_features = [
    "Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType",
]

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("weekend", "passthrough", ["Weekend"]),
])

pipeline = Pipeline([("preprocessing", preprocessor)])